In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

Pipeline de trabalho proposta:
- Coleta (baixar planilhas da estação aracaju A409) no periodo 2016 - 2025 diretamente do site da IMNET
- integração dos dados históricos
- Análise da disponibilidade da base: columns (shape e tipos), dados faltantes, duplicatas
- transformar colunas de data e numéricas
- agregação para uso em séries temporais
- padronização

In [2]:
pd_a409_2016 = pd.read_csv('raw_data/aracaju_A409_2016.csv', sep=';')
pd_a409_2016.head()

,Data,Hora (UTC),Temp. Ins. (C),Temp. Max. (C),Temp. Min. (C),Umi. Ins. (%),Umi. Max. (%),Umi. Min. (%),Pto Orvalho Ins. (C),Pto Orvalho Max. (C),Pto Orvalho Min. (C),Pressao Ins. (hPa),Pressao Max. (hPa),Pressao Min. (hPa),Vel. Vento (m/s),Dir. Vento (m/s),Raj. Vento (m/s),Radiacao (KJ/m²),Chuva (mm)
0,01/01/2016,0,"27,0","27,1","27,0","66,0","67,0","64,0","20,0","20,5","19,8","1013,5","1013,6","1013,0","4,6","81,0","8,7",NaN,"0,0"
1,01/01/2016,100,"26,9","27,0","26,9","65,0","66,0","63,0","19,7","20,2","19,4","1013,6","1013,6","1013,4","4,4","85,0","8,3",NaN,"0,0"
2,01/01/2016,200,"26,8","27,0","26,8","65,0","66,0","62,0","19,6","19,8","19,0","1013,1","1013,6","1013,1","5,2","81,0","8,8",NaN,"0,0"
3,01/01/2016,300,"26,7","26,8","26,6","66,0","66,0","65,0","19,7","19,9","19,5","1012,6","1013,1","1012,6","4,7","79,0","8,1",NaN,"0,0"
4,01/01/2016,400,"26,6","26,7","26,5","66,0","67,0","65,0","19,6","19,9","19,5","1012,1","1012,6","1012,1","4,1","78,0","7,8",NaN,"0,0"


### Integração das tabelas

In [3]:
time_range = range(2016,2026)
dfs = []

for year in time_range:
  df = pd.read_csv(f"raw_data/aracaju_A409_{year}.csv", sep=";")
  dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

df_historico = df.copy()

df_historico.head()

,Data,Hora (UTC),Temp. Ins. (C),Temp. Max. (C),Temp. Min. (C),Umi. Ins. (%),Umi. Max. (%),Umi. Min. (%),Pto Orvalho Ins. (C),Pto Orvalho Max. (C),Pto Orvalho Min. (C),Pressao Ins. (hPa),Pressao Max. (hPa),Pressao Min. (hPa),Vel. Vento (m/s),Dir. Vento (m/s),Raj. Vento (m/s),Radiacao (KJ/m²),Chuva (mm)
0,01/01/2016,0,"27,0","27,1","27,0","66,0","67,0","64,0","20,0","20,5","19,8","1013,5","1013,6","1013,0","4,6","81,0","8,7",NaN,"0,0"
1,01/01/2016,100,"26,9","27,0","26,9","65,0","66,0","63,0","19,7","20,2","19,4","1013,6","1013,6","1013,4","4,4","85,0","8,3",NaN,"0,0"
2,01/01/2016,200,"26,8","27,0","26,8","65,0","66,0","62,0","19,6","19,8","19,0","1013,1","1013,6","1013,1","5,2","81,0","8,8",NaN,"0,0"
3,01/01/2016,300,"26,7","26,8","26,6","66,0","66,0","65,0","19,7","19,9","19,5","1012,6","1013,1","1012,6","4,7","79,0","8,1",NaN,"0,0"
4,01/01/2016,400,"26,6","26,7","26,5","66,0","67,0","65,0","19,6","19,9","19,5","1012,1","1012,6","1012,1","4,1","78,0","7,8",NaN,"0,0"


### Diagnóstico Inicial

In [4]:
df_historico.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87624 entries, 0 to 87623
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Data                  87624 non-null  object
 1   Hora (UTC)            87624 non-null  int64 
 2   Temp. Ins. (C)        68917 non-null  object
 3   Temp. Max. (C)        66446 non-null  object
 4   Temp. Min. (C)        68913 non-null  object
 5   Umi. Ins. (%)         62669 non-null  object
 6   Umi. Max. (%)         62600 non-null  object
 7   Umi. Min. (%)         62590 non-null  object
 8   Pto Orvalho Ins. (C)  54314 non-null  object
 9   Pto Orvalho Max. (C)  54239 non-null  object
 10  Pto Orvalho Min. (C)  54231 non-null  object
 11  Pressao Ins. (hPa)    77220 non-null  object
 12  Pressao Max. (hPa)    77258 non-null  object
 13  Pressao Min. (hPa)    77258 non-null  object
 14  Vel. Vento (m/s)      65597 non-null  object
 15  Dir. Vento (m/s)      73470 non-null

In [5]:
df_historico = df_historico.rename(columns={
    "Data": "data",
    "Hora (UTC)": "hora_utc",
    "Temp. Ins. (C)": "temp_inst",
    "Temp. Max. (C)": "temp_max",
    "Temp. Min. (C)": "temp_min",
    "Umi. Ins. (%)": "umidade_inst",
    "Umi. Max. (%)": "umidade_max",
    "Umi. Min. (%)": "umidade_min",
    "Pto Orvalho Ins. (C)": "orvalho_inst",
    "Pto Orvalho Max. (C)": "orvalho_max",
    "Pto Orvalho Min. (C)": "orvalho_min",
    "Pressao Ins. (hPa)": "pressao_inst",
    "Pressao Max. (hPa)": "pressao_max",
    "Pressao Min. (hPa)": "pressao_min",
    "Vel. Vento (m/s)": "vento_vel",
    "Dir. Vento (m/s)": "vento_dir",
    "Raj. Vento (m/s)": "vento_raj",
    "Radiacao (KJ/m²)": "radiacao",
    "Chuva (mm)": "chuva"
})

float_columns = df_historico.iloc[:,2:]


for c in float_columns:
  df_historico[c] = (
      df_historico[c]
      .astype(str)
      .str.replace(",", ".")
      .replace("", pd.NA)
  )
  df_historico[c] = pd.to_numeric(df_historico[c], errors='coerce')

df_historico

,data,hora_utc,temp_inst,temp_max,temp_min,umidade_inst,umidade_max,umidade_min,orvalho_inst,orvalho_max,orvalho_min,pressao_inst,pressao_max,pressao_min,vento_vel,vento_dir,vento_raj,radiacao,chuva
0,01/01/2016,0,27.0,27.1,27.0,66.0,67.0,64.0,20.0,20.5,19.8,1013.5,1013.6,1013.0,4.6,81.0,8.7,NaN,0.0
1,01/01/2016,100,26.9,27.0,26.9,65.0,66.0,63.0,19.7,20.2,19.4,1013.6,1013.6,1013.4,4.4,85.0,8.3,NaN,0.0
2,01/01/2016,200,26.8,27.0,26.8,65.0,66.0,62.0,19.6,19.8,19.0,1013.1,1013.6,1013.1,5.2,81.0,8.8,NaN,0.0
3,01/01/2016,300,26.7,26.8,26.6,66.0,66.0,65.0,19.7,19.9,19.5,1012.6,1013.1,1012.6,4.7,79.0,8.1,NaN,0.0
4,01/01/2016,400,26.6,26.7,26.5,66.0,67.0,65.0,19.6,19.9,19.5,1012.1,1012.6,1012.1,4.1,78.0,7.8,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87619,31/12/2025,1900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
87620,31/12/2025,2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
87621,31/12/2025,2100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
87622,31/12/2025,2200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Tratamento

In [6]:

# Tratando hora
df_historico["hora_utc"] = (df_historico["hora_utc"]
                .astype(str)
                .str.zfill(4)
)

# Criando timestamp completo
df_historico["timestamp"] = pd.to_datetime(
    df_historico["data"] + " " + df_historico["hora_utc"],
    format="%d/%m/%Y %H%M"
)

df_historico['data'] = pd.to_datetime(df_historico["data"], format='%d/%m/%Y')
df_historico['ano'] = df_historico['data'].dt.year
df_historico['mes'] = df_historico['data'].dt.month



### Tratamento de valores faltantes

In [7]:
df_historico = df_historico.sort_values("timestamp").copy()

df_historico["chuva"] = df_historico["chuva"].fillna(0.0)

df_historico

,data,hora_utc,temp_inst,temp_max,temp_min,umidade_inst,umidade_max,umidade_min,orvalho_inst,orvalho_max,...,pressao_max,pressao_min,vento_vel,vento_dir,vento_raj,radiacao,chuva,timestamp,ano,mes
0,2016-01-01,0000,27.0,27.1,27.0,66.0,67.0,64.0,20.0,20.5,...,1013.6,1013.0,4.6,81.0,8.7,NaN,0.0,2016-01-01 00:00:00,2016,1
1,2016-01-01,0100,26.9,27.0,26.9,65.0,66.0,63.0,19.7,20.2,...,1013.6,1013.4,4.4,85.0,8.3,NaN,0.0,2016-01-01 01:00:00,2016,1
2,2016-01-01,0200,26.8,27.0,26.8,65.0,66.0,62.0,19.6,19.8,...,1013.6,1013.1,5.2,81.0,8.8,NaN,0.0,2016-01-01 02:00:00,2016,1
3,2016-01-01,0300,26.7,26.8,26.6,66.0,66.0,65.0,19.7,19.9,...,1013.1,1012.6,4.7,79.0,8.1,NaN,0.0,2016-01-01 03:00:00,2016,1
4,2016-01-01,0400,26.6,26.7,26.5,66.0,67.0,65.0,19.6,19.9,...,1012.6,1012.1,4.1,78.0,7.8,NaN,0.0,2016-01-01 04:00:00,2016,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87619,2025-12-31,1900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2025-12-31 19:00:00,2025,12
87620,2025-12-31,2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2025-12-31 20:00:00,2025,12
87621,2025-12-31,2100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2025-12-31 21:00:00,2025,12
87622,2025-12-31,2200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2025-12-31 22:00:00,2025,12


In [8]:
df_historico.head()


,data,hora_utc,temp_inst,temp_max,temp_min,umidade_inst,umidade_max,umidade_min,orvalho_inst,orvalho_max,...,pressao_max,pressao_min,vento_vel,vento_dir,vento_raj,radiacao,chuva,timestamp,ano,mes
0,2016-01-01,0000,27.0,27.1,27.0,66.0,67.0,64.0,20.0,20.5,...,1013.6,1013.0,4.6,81.0,8.7,NaN,0.0,2016-01-01 00:00:00,2016,1
1,2016-01-01,0100,26.9,27.0,26.9,65.0,66.0,63.0,19.7,20.2,...,1013.6,1013.4,4.4,85.0,8.3,NaN,0.0,2016-01-01 01:00:00,2016,1
2,2016-01-01,0200,26.8,27.0,26.8,65.0,66.0,62.0,19.6,19.8,...,1013.6,1013.1,5.2,81.0,8.8,NaN,0.0,2016-01-01 02:00:00,2016,1
3,2016-01-01,0300,26.7,26.8,26.6,66.0,66.0,65.0,19.7,19.9,...,1013.1,1012.6,4.7,79.0,8.1,NaN,0.0,2016-01-01 03:00:00,2016,1
4,2016-01-01,0400,26.6,26.7,26.5,66.0,67.0,65.0,19.6,19.9,...,1012.6,1012.1,4.1,78.0,7.8,NaN,0.0,2016-01-01 04:00:00,2016,1


Considerando que há uma disparidade entre as parcelas de valores faltantes para cada tipo de coluna, o mais adequado seria adotar uma estratégia diferente para resolver a questão dos dados faltantes.

In [9]:
missing_percent = (df_historico.isna().mean() * 100).sort_values(ascending=False)

display(missing_percent)

radiacao        52.049667
orvalho_min     38.109422
orvalho_max     38.100292
orvalho_inst    38.014699
umidade_min     28.569798
umidade_max     28.558386
umidade_inst    28.479640
vento_raj       25.142655
vento_vel       25.138090
temp_max        24.169177
temp_min        21.353739
temp_inst       21.349174
vento_dir       16.153109
pressao_inst    11.873459
pressao_max     11.830092
pressao_min     11.830092
timestamp        0.000000
ano              0.000000
data             0.000000
chuva            0.000000
hora_utc         0.000000
mes              0.000000
dtype: float64

In [10]:
missing_by_year = (df_historico.isna()
  .groupby(df_historico["ano"])
  .mean()
  .mul(100)
)
missing_by_year

,data,hora_utc,temp_inst,temp_max,temp_min,umidade_inst,umidade_max,umidade_min,orvalho_inst,orvalho_max,...,pressao_max,pressao_min,vento_vel,vento_dir,vento_raj,radiacao,chuva,timestamp,ano,mes
ano,,,,,,,,,,,,,,,,,,,,,
2016,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,60.136986,0.000000,60.136986,45.650685,0.0,0.0,0.0,0.0
2017,0.0,0.0,1.156136,1.179029,1.179029,1.156136,1.179029,1.179029,1.156136,1.179029,...,1.179029,1.179029,74.381868,44.562729,74.404762,45.650183,0.0,0.0,0.0,0.0
2018,0.0,0.0,0.000000,0.011416,0.011416,0.000000,0.011416,0.011416,0.000000,0.011416,...,0.011416,0.011416,0.000000,0.000000,0.011416,45.605023,0.0,0.0,0.0,0.0
2019,0.0,0.0,0.000000,0.011416,0.011416,32.283105,32.305936,32.305936,32.283105,32.328767,...,0.011416,0.011416,0.000000,0.000000,0.011416,44.257991,0.0,0.0,0.0,0.0
2020,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,46.402550,0.0,0.0,0.0,0.0
2021,0.0,0.0,7.796804,35.958904,7.796804,62.842466,62.876712,62.876712,62.853881,62.876712,...,7.899543,7.899543,7.899543,7.899543,7.899543,50.319635,0.0,0.0,0.0,0.0
2022,0.0,0.0,69.817352,69.817352,69.817352,85.878995,86.073059,86.073059,85.878995,86.084475,...,69.817352,69.817352,69.817352,69.817352,69.817352,83.515982,0.0,0.0,0.0,0.0
2023,0.0,0.0,14.109589,14.109589,14.109589,77.408676,77.910959,78.025114,77.408676,77.956621,...,14.109589,14.109589,14.109589,14.109589,14.109589,53.310502,0.0,0.0,0.0,0.0
2024,0.0,0.0,20.617031,20.617031,20.617031,0.056922,0.056922,0.056922,20.617031,20.617031,...,0.056922,0.056922,0.056922,0.056922,0.056922,45.833333,0.0,0.0,0.0,0.0


In [17]:
df_mensal = (
    df_historico
    .set_index("timestamp")
    .resample("1M")
    .agg({
        "temp_inst": "mean",
        "umidade_inst": "mean",
        "pressao_inst": "mean",
        "vento_vel": "mean",
        "vento_raj": "max",
        "chuva": "sum",
        "radiacao": "sum"
    })
)

df_mensal.index = df_mensal.index.strftime('%m/%Y')
df_mensal.index.name = 'Mes/Ano'
df_mensal

,temp_inst,umidade_inst,pressao_inst,vento_vel,vento_raj,chuva,radiacao
Mes/Ano,,,,,,,
01/2016,27.439247,70.342742,1012.329167,3.007930,12.2,83.0,544224.9
02/2016,28.341954,63.129310,1013.361782,2.884626,13.5,0.0,614982.0
03/2016,28.610484,62.819892,1013.512769,2.752554,12.7,0.0,608876.9
04/2016,28.335139,62.081944,1013.212639,2.623889,14.9,0.0,525674.1
05/2016,27.036559,66.172043,1015.263441,1.843027,12.4,206.0,452149.9
...,...,...,...,...,...,...,...
08/2025,NaN,70.846361,1017.524528,1.811321,11.3,204.6,473315.7
09/2025,NaN,69.025641,1017.461538,2.135897,10.7,93.8,505923.2
10/2025,NaN,67.784615,1018.389231,1.383077,9.8,4.0,33564.6


In [32]:
df_mensal.to_csv("processed_data/aracaju_A409_mensal.csv", index=False, sep=",")

In [19]:
df_historico.to_csv("processed_data/aracaju_A409.csv", index=False, sep=",")

os próximos passos devem consistir em criar análises derivadas por horario, dia, mes e ano



considerando a heterogeneidade na série histórica, onde o perfil de dados nulos para cada ano varia bastante dentro da série, se torna relevante aplicar métodos distintos para normalização da base, a proposta de tratamento baseado na porcentagem de dados faltantes segue a descrição abaixo:
- loss(0.0% - 1.0%): interpolação linear simples para tapar "buracos"
- loss(1.0% - 20.0%): interpolação limitada a pequenos intervalos aplicada juntamente com a média do mês/horário para janelas de falha maiores

In [22]:
minimum_loss_df = df_historico[df_historico["ano"].isin([2016, 2018, 2020])].copy()

numeric_cols = minimum_loss_df.select_dtypes(include=["float64"]).columns
minimum_loss_df[numeric_cols] = minimum_loss_df[numeric_cols].interpolate(method="linear")


In [23]:
missing_by_year_2 = (minimum_loss_df.isna()
  .groupby(minimum_loss_df["ano"])
  .mean()
  .mul(100)
)
missing_by_year_2

,data,hora_utc,temp_inst,temp_max,temp_min,umidade_inst,umidade_max,umidade_min,orvalho_inst,orvalho_max,...,pressao_max,pressao_min,vento_vel,vento_dir,vento_raj,radiacao,chuva,timestamp,ano,mes
ano,,,,,,,,,,,,,,,,,,,,,
2016,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.10274,0.0,0.0,0.0,0.0
2018,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0
2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0


In [28]:
medium_loss_df = df_historico[df_historico["ano"].isin([2017, 2019, 2021, 2024])].copy()

cols_continuas = ["temp_inst", "temp_max", "temp_min", "umidade_inst", 
                  "umidade_max", "umidade_min", "orvalho_inst", "orvalho_max", 
                  "orvalho_min", "pressao_inst", "pressao_max", "pressao_min"]

# Preenchendo os gaps nulos menores com interpolação linear
medium_loss_df[cols_continuas] = medium_loss_df[cols_continuas].interpolate(
    method="linear", limit=3, limit_direction="forward"
)

# Para os intervalos maiores a estratégia adotada será de aplicar média
medium_loss_df["hora"] = medium_loss_df["timestamp"].dt.hour
medium_loss_df[cols_continuas] = medium_loss_df.groupby(["mes", "hora"])[cols_continuas].transform(
    lambda group: group.fillna(group.mean())
)

medium_loss_df.drop(columns=["hora"], inplace=True)

In [30]:
missing_values_3 = (medium_loss_df.isna()
                    .groupby(medium_loss_df["ano"])
                    .mean()
                    .mul(100)
                   )
missing_values_3

,data,hora_utc,temp_inst,temp_max,temp_min,umidade_inst,umidade_max,umidade_min,orvalho_inst,orvalho_max,...,pressao_max,pressao_min,vento_vel,vento_dir,vento_raj,radiacao,chuva,timestamp,ano,mes
ano,,,,,,,,,,,,,,,,,,,,,
2017,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,74.381868,44.562729,74.404762,45.650183,0.0,0.0,0.0,0.0
2019,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.000000,0.000000,0.011416,44.257991,0.0,0.0,0.0,0.0
2021,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,7.899543,7.899543,7.899543,50.319635,0.0,0.0,0.0,0.0
2024,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.056922,0.056922,0.056922,45.833333,0.0,0.0,0.0,0.0


In [35]:
df_not_treated = df_historico[df_historico["ano"].isin([2022, 2023, 2025])].copy()

df_complete = (
    pd.concat([minimum_loss_df, medium_loss_df, df_not_treated])
    .sort_values("timestamp")
    .reset_index(drop=True)
)

df_mensal_2 = (df_complete
            .set_index("timestamp")
            .resample("1M")
            .agg({
                "temp_inst": "mean",
                "chuva": "sum"
            })
)


df_mensal_2.index = df_mensal_2.index.strftime('%m/%Y')
df_mensal_2.index.name = 'Mes/Ano'
df_mensal_2

,temp_inst,chuva
Mes/Ano,,
01/2016,27.439247,83.0
02/2016,28.341954,0.0
03/2016,28.610484,0.0
04/2016,28.335139,0.0
05/2016,27.036559,206.0
...,...,...
08/2025,NaN,204.6
09/2025,NaN,93.8
10/2025,NaN,4.0


In [36]:
df_result = (
    pd.concat([minimum_loss_df, medium_loss_df])
    .query("ano not in [2022, 2023, 2025]")
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print("Missing values by column")
print(df_result.isna().sum())

Missing values by column
data                0
hora_utc            0
temp_inst           0
temp_max            0
temp_min            0
umidade_inst        0
umidade_max         0
umidade_min         0
orvalho_inst        0
orvalho_max         0
orvalho_min         0
pressao_inst        0
pressao_max         0
pressao_min         0
vento_vel        7195
vento_dir        4590
vento_raj        7198
radiacao        16308
chuva               0
timestamp           0
ano                 0
mes                 0
dtype: int64


In [37]:
df_result.to_csv("processed_data/aracaju_A409_filtered.csv", index=False, sep=",")